##  Projet Kayak - Moteur de Recommandation de Destinations

**Contexte** : Développer une application qui recommande les meilleures destinations
de vacances en France basée sur les données météo et hôtels
 
**Livrables** :
 - CSV enrichi dans un bucket S3
 - Base de données SQL avec les données nettoyées
 - Cartes interactives Top-5 destinations et Top-20 hôtels
 
**Stack technique** : Python, OpenMeteo API, Overpass API, AWS S3/RDS, Plotly

## Imports

In [ ]:
import requests
import pandas as pd
import time
import json
import os
import random
import plotly.express as px
import unicodedata
from dotenv import load_dotenv
from bs4 import BeautifulSoup
import boto3
from sqlalchemy import create_engine, text


In [ ]:
load_dotenv()

class SecureCloudManager:
    def __init__(self):
        self.s3_client = boto3.client(
            's3',
            aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
            aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
            region_name=os.getenv('AWS_REGION', 'eu-west-3')
        )

## Config & cache

In [ ]:

CITIES = [
    "Mont Saint Michel","St Malo","Bayeux","Le Havre","Rouen","Paris",
    "Amiens","Lille","Strasbourg","Chateau du Haut Koenigsbourg","Colmar",
    "Eguisheim","Besancon","Dijon","Annecy","Grenoble","Lyon",
    "Gorges du Verdon","Bormes les Mimosas","Cassis","Marseille",
    "Aix en Provence","Avignon","Uzes","Nimes","Aigues Mortes",
    "Saintes Maries de la mer","Collioure","Carcassonne","Ariege",
    "Toulouse","Montauban","Biarritz","Bayonne","La Rochelle"
]
SPECIAL_OVERRIDES = {
    "St Malo": "Saint-Malo",
    "Chateau du Haut Koenigsbourg": "Orschwiller",
    "Gorges du Verdon": "La Palud-sur-Verdon",
    "Ariege": "Foix",
    "Mont Saint Michel": "Le Mont-Saint-Michel"
    }
HOTEL_SEARCH_RADIUS_M=10000
CACHE_PATH="hotels_cache.json"

import json
import os

def load_hotels_cache(path):
    if os.path.exists(path):
        try:
            with open(path, "r") as f:
                return json.load(f)
        except Exception as e:
            print(f"[WARN] Failed to load cache: {e}. Starting with empty cache.")
            return {}
    else:
        return {}

HOTELS_CACHE = load_hotels_cache(CACHE_PATH)


## Booking.com website scraping

In [ ]:
def scrape_booking_hotels(city_name, max_hotels=20):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }

    query = f"{city_name} hotels"
    url = f"https://www.booking.com/searchresults.fr.html?ss={query}"

    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, 'html.parser')

        hotels = []

        for card in soup.select('div[data-testid="property-card"]'):
            if len(hotels) >= max_hotels:
                break

            name = card.select_one('div[data-testid="title"]').get_text(strip=True)
            url = card.select_one('a[data-testid="title-link"]')['href']
            rating = card.select_one('div[data-testid="review-score"]')
            description = card.select_one('div[data-testid="description"]')

            hotels.append({
                'hotel_name': name,
                'hotel_url': f"https://www.booking.com{url}",
                'hotel_rating': float(rating.get_text(strip=True)) if rating else None,
                'hotel_description': description.get_text(strip=True) if description else None
            })

        time.sleep(2 + random.random())
        return hotels

    except Exception as e:
        print(e)
        return []


# ============ GEOCODING OPEN-METEO ============

In [ ]:
def normalize_city_name(name: str):
    """Removes accents, multiple spaces, standardises the name."""
    name = "".join(c for c in name if unicodedata.category(c) != 'Mn')
    name = " ".join(name.split())
    return name


def geocode_city(city_name: str, country: str = "France"):
    """
    Geocoding ultra-robuste pour Open-Meteo :
    - corrige les noms complexes via SPECIAL_OVERRIDES
    - normalise les accents
    - teste plusieurs variantes
    """
    base_url = "https://geocoding-api.open-meteo.com/v1/search"

    # 1) Correction of names not found
    adjusted_name = SPECIAL_OVERRIDES.get(city_name, city_name)

    # 2) Normalisation
    formatted = normalize_city_name(adjusted_name)

    # 3) Tentatives successives
    attempts = [
        f"{formatted}, {country}",
        formatted,
        adjusted_name,
        adjusted_name.replace(" ", "-"),
        adjusted_name.replace(" ", ""),
    ]

    for q in attempts:
        try:
            resp = requests.get(base_url, params={"name": q, "count": 1, "language": "fr"})
            resp.raise_for_status()
            results = resp.json().get("results")
            if results:
                r = results[0]
                return {
                    "city": city_name,   # on garde le nom original
                    "lat": r["latitude"],
                    "lon": r["longitude"]
                }
        except:
            pass

    print(f"[WARN] Impossible de géocoder : {city_name}")
    return None


def build_cities_geo_df():
    """Construit le dataframe final pour toutes les villes."""
    rows = []
    for i, c in enumerate(CITIES, start=1):
        g = geocode_city(c)
        if g:
            g["city_id"] = i
            rows.append(g)
        time.sleep(0.5)

    return pd.DataFrame(rows)[["city_id", "city", "lat", "lon"]]

In [ ]:
df_geo = build_cities_geo_df()
df_geo

In [ ]:
# Weather
def compute_score(w):
    """
    Compute a nice weather score based on the forecast.
    Higher score means nicer weather (higher temps, lower rain).
    """
    daily = w.get('daily', {})
    temps = daily.get('temperature_2m_max', [])
    rains = daily.get('precipitation_sum', [])
    
    if not temps or not rains:
        return 0.0
    
    avg_temp = sum(temps) / len(temps)
    total_rain = sum(rains)
    
    # Simple score: average temp minus scaled rain (to keep positive-ish)
    score = avg_temp - total_rain * 0.1
    return round(score, 2)

def fetch_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": [
            "temperature_2m_min",
            "temperature_2m_max",
            "precipitation_sum",
            "precipitation_probability_mean"
        ],
        "timezone": "Europe/Paris"
    }

    # Timeout + retry
    for attempt in range(3):
        try:
            r = requests.get(url, params=params, timeout=5)   # IMPORTANT
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"[WARN] Weather retry {attempt+1}/3 at ({lat},{lon}) : {e}")
            time.sleep(0.3)

    print(f"[ERROR] Weather failed for ({lat},{lon})")
    return None

def build_weather_df(df):
    rows=[]
    for _, r in df.iterrows():
        w = fetch_weather(r.lat, r.lon)
        if w is None:
            continue

        rows.append({
            "city_id": r.city_id,
            "city": r.city,
            "lat": r.lat,
            "lon": r.lon,
            "nice_weather_score": compute_score(w),
            "weather_raw": json.dumps(w)
        })

        time.sleep(0.2)   # réduit

    return pd.DataFrame(rows)



In [ ]:
df_weather = build_weather_df(df_geo)
df_weather

## ======Animated map  J1 → J7

In [ ]:
# Ensure marker size is non-negative
def plot_weather_animated_fixed(df_weather, html_path="weather_forecast_animated.html"):
	"""
	Animated map (D1 → D7) of the maximum daily temperature.
	"""
	rows = []

	for _, r in df_weather.iterrows():
		raw = json.loads(r["weather_raw"])
		temps = raw["daily"]["temperature_2m_max"]
		rains = raw["daily"]["precipitation_sum"]
		probs = raw["daily"]["precipitation_probability_mean"]

		for day in range(len(temps)):
			rows.append({
				"city": r.city,
				"lat": r.lat,
				"lon": r.lon,
				"day": day + 1,      # D+1 → D+7
				"temp": temps[day],
				"rain_mm": rains[day],
				"rain_prob": probs[day],
				"nice_score": r.nice_weather_score
			})

	df = pd.DataFrame(rows)
	# Clip negative values to zero for marker size
	df["nice_score_plot"] = df["nice_score"].clip(lower=0)
	size="nice_score_plot"

	fig = px.scatter_map(
		df,
		lat="lat",
		lon="lon",
		animation_frame="day",
		color="temp",
		size=size,
		hover_name="city",
		hover_data=["temp", "rain_mm", "rain_prob"],
		zoom=5,
		height=700
	)

	fig.update_layout(mapbox_style="open-street-map")
	fig.update_layout(title="Weather animation J1 → J7")

	fig.write_html(html_path)
	print(f"[INFO] Animated map saved : {html_path}")

	return fig

plot_weather_animated_fixed(df_weather)

## ====2) Filterable map: temperature / rainfall / score

In [ ]:
def plot_weather_filter(df_weather, metric="temp", html_path="weather_filter_map.html"):
    """
    Carte filtrable par :
    - temp : température max J+7
    - rain : pluie J+7 (mm)
    - score : weather score personnalisé
    """
    rows = []

    for _, r in df_weather.iterrows():
        raw = json.loads(r["weather_raw"])
        daily = raw["daily"]

        try:
            j7_temp = daily["temperature_2m_max"][6]
            j7_rain = daily["precipitation_sum"][6]
            j7_prob = daily["precipitation_probability_mean"][6]
        except:
            continue

        rows.append({
            "city": r.city,
            "lat": r.lat,
            "lon": r.lon,
            "temp": j7_temp,
            "rain": j7_rain,
            "score": r.nice_weather_score
        })

    df = pd.DataFrame(rows)

    if metric not in ["temp", "rain", "score"]:
        raise ValueError("metric doit être : temp | rain | score")

    fig = px.scatter_map(
        df,
        lat="lat",
        lon="lon",
        color="metric",
        size="score",
        hover_name="city",
        hover_data=["temp", "rain", "score"],
        zoom=5,
        height=700
    )

    fig.update_layout(mapbox_style="open-street-map")
    fig.update_layout(title=f"Filtered map by {metric}")

    fig.write_html(html_path)
    print(f"[INFO] Filtered map saved : {html_path}")

    return fig

In [ ]:
df = df_weather.copy()
df["score_size"] = df["nice_weather_score"].clip(lower=0)

fig = px.scatter_map(
    df, lat="lat", lon="lon",
    color="nice_weather_score",
    size="score_size",
    hover_name="city",
    hover_data=["nice_weather_score"],
    zoom=5, height=700
)


In [ ]:
# Ensure the size column is non-negative for all metrics
df_weather["score_size"] = df_weather["nice_weather_score"].clip(lower=0)

fig = px.scatter_mapbox(
	df_weather,
	lat="lat",
	lon="lon",
	color="nice_weather_score",
	size="score_size",
	hover_name="city",
	hover_data=["nice_weather_score"],
	zoom=5,
	height=700
)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(title="Weather Forecast with Metrics")
fig.show()

## === Map combining weather and hotels

In [ ]:
def plot_weather_hotels(df_weather, df_hotels, html_path="weather_hotels_map.html"):
    """
    Combine:
        - 7-day weather forecast for cities
        - hotels (secondary points)
    """
    # --- 1) Recovery on day 7 for cities ---
    weather_info = {}
    for _, r in df_weather.iterrows():
        raw = json.loads(r["weather_raw"])
        try:
            weather_info[r.city_id] = {
                "temp_j7": raw["daily"]["temperature_2m_max"][6],
                "rain_j7": raw["daily"]["precipitation_sum"][6],
                "prob_j7": raw["daily"]["precipitation_probability_mean"][6],
                "lat": r.lat,
                "lon": r.lon,
                "city": r.city
            }
        except:
            pass

    # --- 2) Building the enriched hotel dataframe ---
    records = []
    for _, h in df_hotels.iterrows():
        info = weather_info.get(h.city_id)
        if not info:
            continue

        records.append({
            "hotel_name": h.hotel_name,
            "hotel_lat": h.hotel_lat if h.hotel_lat else info["lat"],
            "hotel_lon": h.hotel_lon if h.hotel_lon else info["lon"],
            "city": info["city"],
            "temp_j7": info["temp_j7"],
            "rain_j7": info["rain_j7"],
            "prob_j7": info["prob_j7"],
        })

    df = pd.DataFrame(records)

    fig = px.scatter_map(
        df,
        lat="hotel_lat",
        lon="hotel_lon",
        color="temp_j7",
        size_max=12,
        hover_name="hotel_name",
        hover_data=["city", "temp_j7", "rain_j7", "prob_j7"],
        zoom=5,
        height=700
    )

    fig.update_layout(mapbox_style="open-street-map")
    fig.update_layout(title="Hôtels + météo J+7")

    fig.write_html(html_path)
    print(f"[INFO] Carte combinée sauvegardée : {html_path}")

    return fig


In [ ]:
df_weather["weather_raw"].apply(lambda x: len(json.loads(x)["daily"]["temperature_2m_max"]))

In [ ]:
df_weather["weather_raw"].apply(lambda x: json.loads(x)["daily"]["temperature_2m_max"][6])

In [ ]:
def plot_weather_forecast_map(df_weather, html_path="weather_forecast_map.html"):
    """
    Plot a map showing the nice weather scores for each city.
    """
    fig = px.scatter_map(
        df_weather,
        lat="lat",
        lon="lon",
        color="nice_weather_score",
        hover_name="city",
        hover_data=["nice_weather_score"],
        zoom=5,
        height=700
    )
    fig.update_layout(mapbox_style="open-street-map")
    fig.update_layout(title="Weather Forecast Map")
    fig.write_html(html_path)
    print(f"[INFO] Saved map : {html_path}")
    return fig

plot_weather_forecast_map(df_weather)

## Save & map cities

In [ ]:
def save_weather_csv(df,path="weather_cities.csv"):
    df.to_csv(path,index=False)

def plot_top_cities(df,n=5,path="top5_cities.html"):
    top=df.sort_values("nice_weather_score",ascending=False).head(n)
    fig=px.scatter_map(top,lat="lat",lon="lon",hover_name="city",
                          hover_data=["nice_weather_score"],zoom=4,height=600)
    fig.update_layout(mapbox_style="open-street-map")
    fig.write_html(path)


In [ ]:
df_weather = build_weather_df(df_geo)
df_weather

## Booking Scraping

In [ ]:
def scrape_booking_hotels(city_name, max_hotels=20):
    """Scraping Booking.com conforme au cahier des charges"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    query = f"{city_name} hotels"
    url = f"https://www.booking.com/searchresults.fr.html?ss={query}"

    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        hotels = []
        for card in soup.select('div[data-testid="property-card"]'):
            if len(hotels) >= max_hotels:
                break

            name_tag = card.select_one('div[data-testid="title"]')
            link_tag = card.select_one('a[data-testid="title-link"]')
            rating_tag = card.select_one('div[data-testid="review-score"]')
            desc_tag = card.select_one('div[data-testid="description"]')

            if not name_tag or not link_tag:
                continue

            hotels.append({
                "hotel_name": name_tag.get_text(strip=True),
                "hotel_url": "https://www.booking.com" + link_tag["href"],
                "hotel_rating": float(rating_tag.get_text(strip=True)) if rating_tag else None,
                "hotel_description": desc_tag.get_text(strip=True) if desc_tag else None,
            })

        time.sleep(2 + random.random())
        return hotels

    except Exception as e:
        print(f"Erreur de scraping pour {city_name} : {e}")
        return []


## Overpass hotels with retry+cache  
if previous scraping fails.

In [ ]:

TYPES_OK={"hotel","motel","hostel"}

def fetch_hotels(lat,lon,radius=HOTEL_SEARCH_RADIUS_M,retries=3):
    key=f"{lat:.4f}_{lon:.4f}_{radius}"
    if key in HOTELS_CACHE:
        return HOTELS_CACHE[key]
    query=f"""
    [out:json];
    (
      node["tourism"="hotel"](around:{radius},{lat},{lon});
      way["tourism"="hotel"](around:{radius},{lat},{lon});
      relation["tourism"="hotel"](around:{radius},{lat},{lon});
    );
    out center;
    """
    url="https://overpass-api.de/api/interpreter"
    for a in range(retries):
        try:
            r=requests.get(url,params={"data":query},timeout=30)
            r.raise_for_status()
            data=r.json()
            hotels=[]
            for el in data.get("elements",[]):
                t=el.get("tags",{})
                if t.get("tourism") not in TYPES_OK: continue
                name=t.get("name")
                if not name: continue
                la=el.get("lat") or el.get("center",{}).get("lat")
                lo=el.get("lon") or el.get("center",{}).get("lon")
                if not la or not lo: continue
                hotels.append({
                    "hotel_name":name,"hotel_lat":la,"hotel_lon":lo,
                    "hotel_desc":t.get("description"),"hotel_website":t.get("website"),
                    "hotel_phone":t.get("phone"),"hotel_stars":t.get("stars")
                })
            HOTELS_CACHE[key]=hotels
            json.dump(HOTELS_CACHE,open(CACHE_PATH,"w"),indent=2)
            return hotels
        except:
            time.sleep(1*(2**a)+random.random())
    return []

def build_hotels_df(df,max_hotels=20):
    rec=[]; failed=[]
    for _,r in df.iterrows():
        h=fetch_hotels(r.lat,r.lon)
        if not h:
            failed.append(r.city); continue
        h=h[:max_hotels]
        for x in h:
            x["city"]=r.city; x["city_id"]=r.city_id
            rec.append(x)
        time.sleep(0.5)
    for city in failed:
        rw=df[df.city==city].iloc[0]
        h=fetch_hotels(rw.lat,rw.lon,radius=HOTEL_SEARCH_RADIUS_M*1.5)
        h=h[:max_hotels]
        for x in h:
            x["city"]=city; x["city_id"]=rw.city_id
            rec.append(x)
        time.sleep(1)
    return pd.DataFrame(rec)


In [ ]:
df_hotels = build_hotels_df(df_weather)
df_hotels.shape

## Map hotels

In [ ]:
def plot_top_hotels(df_hotels,df_weather,n=20,path="top20_hotels.html"):
    if df_hotels.empty: return
    merged=df_hotels.merge(df_weather[["city_id","nice_weather_score"]],on="city_id")
    top=merged.head(n)
    fig=px.scatter_map(top,lat="hotel_lat",lon="hotel_lon",
                          hover_name="hotel_name",
                          hover_data=["city","nice_weather_score","hotel_stars"],
                          zoom=4,height=600)
    fig.update_layout(mapbox_style="open-street-map")
    fig.write_html(path)


## Pipeline

In [ ]:
def certification_pipeline():
    # 1. Robust geocoding
    df_geo = build_cities_geo_df()
    
    # 2. Weather data
    df_weather = build_weather_df(df_geo)
    
    # 3. Scraping Booking.com 
    df_hotels = scrape_booking_hotels_multiple_cities(CITIES)
    
    # 4. Data Lake S3
    upload_to_s3("weather_cities.csv", "kayak-datalake", "raw/weather.csv")
    upload_to_s3("hotels_booking.csv", "kayak-datalake", "raw/hotels.csv")
    
    # 5. Data Warehouse RDS
    load_to_rds(df_weather, "weather_forecasts")
    load_to_rds(df_hotels, "hotels_info")
    
    # 6. Visualisations finales
    plot_top_cities(df_weather, n=5)
    plot_top_hotels(df_hotels, df_weather, n=20)
    
    return df_geo, df_weather, df_hotels

In [ ]:
def main_pipeline():
    df_geo=build_cities_geo_df(); print(df_geo)
    df_weather=build_weather_df(df_geo); print(df_weather)
    save_weather_csv(df_weather)
    plot_top_cities(df_weather)
    df_hotels=build_hotels_df(df_weather); print(df_hotels)
    df_hotels.to_csv("hotels_osm.csv",index=False)
    plot_top_hotels(df_hotels,df_weather)
    return df_geo,df_weather,df_hotels


In [ ]:
df_geo, df_weather, df_hotels = main_pipeline()

In [ ]:
plot_weather_forecast_map(df_weather)

## Cloud Infrastructure (S3 & RDS)

In [ ]:
# ==========================================
# IMPORTS + ENV
# ==========================================
import os
import socket
import boto3
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()  # charge .env dès le début

# ==========================================
# CONFIGURATION
# ==========================================

# S3 (Data Lake)
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "eu-west-3")
S3_BUCKET_NAME = os.getenv("S3_BUCKET_NAME", "dreipfeltbucketkayak")

# RDS (Aurora PostgreSQL via SSH tunnel local)
# IMPORTANT : le tunnel doit être lancé dans un terminal séparé
DB_HOST = os.getenv("DB_HOST", "127.0.0.1")      # force IPv4 local
DB_PORT = int(os.getenv("DB_PORT", "6544"))
DB_NAME = os.getenv("DB_NAME", "postgres")
DB_USER = os.getenv("DB_USER", "postgres")       # validé par ton test psql
DB_PASSWORD = os.getenv("RDS_PASSWORD")          # dans .env

# (Optionnel mais utile pour messages) endpoint réel RDS
RDS_ENDPOINT = os.getenv(
    "RDS_ENDPOINT",
    "database-1.cluster-cpmkokqkeqbb.eu-west-3.rds.amazonaws.com"
)

# ==========================================
# PREFLIGHT CHECKS
# ==========================================

def preflight_s3():
    if not AWS_ACCESS_KEY or not AWS_SECRET_KEY:
        raise RuntimeError(
            "[PREFLIGHT] AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY manquants. "
            "Vérifie .env et load_dotenv()."
        )

def preflight_db():
    if not DB_PASSWORD:
        raise RuntimeError(
            "[PREFLIGHT] RDS_PASSWORD manquant. Vérifie .env et load_dotenv()."
        )

    # Vérifie que le tunnel répond localement (fail fast)
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(2)
    try:
        s.connect((DB_HOST, DB_PORT))
    except Exception as e:
        raise RuntimeError(
            f"[PREFLIGHT] Tunnel INACTIF: {DB_HOST}:{DB_PORT} injoignable.\n"
            f"➡️ Lance le tunnel SSH vers {RDS_ENDPOINT}:5432 (ex: ssh -N -L {DB_PORT}:{RDS_ENDPOINT}:5432 ...)\n"
            f"Détail: {e}"
        )
    finally:
        s.close()

    # Vérifie que Postgres répond vraiment
    engine_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(engine_url, connect_args={"connect_timeout": 5})
    with engine.connect() as conn:
        conn.execute(text("select 1"))
    print("[PREFLIGHT] OK: tunnel + PostgreSQL accessibles")

# ==========================================
# CLOUD MANAGER
# ==========================================

class CloudManager:
    """Handles S3 uploads and RDS writes (via SSH tunnel)."""

    def __init__(self, aws_access_key, aws_secret_key, region_name=AWS_REGION):
        preflight_s3()
        self.s3_client = boto3.client(
            "s3",
            aws_access_key_id=aws_access_key,
            aws_secret_access_key=aws_secret_key,
            region_name=region_name
        )
        self.bucket_name = S3_BUCKET_NAME

    def upload_to_s3(self, local_file, s3_key):
        try:
            print(f"[CLOUD] Uploading {local_file} to S3 bucket {self.bucket_name}...")
            self.s3_client.upload_file(local_file, self.bucket_name, s3_key)
            print(f"[CLOUD] Success: {local_file} uploaded as {s3_key}")
        except Exception as e:
            raise RuntimeError(f"[ERROR] S3 Upload failed: {e}")

    def upload_to_sql(self, df, table_name):
        # pré-check DB à chaque run (pratique en notebook)
        preflight_db()

        try:
            print(f"[CLOUD] Uploading DataFrame to SQL table: {table_name}...")
            engine_url = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
            engine = create_engine(engine_url, connect_args={"connect_timeout": 5})

            # begin() => transaction + commit auto, plus propre
            with engine.begin() as conn:
                conn.execute(text("select 1"))  # preuve de vie DB
                df.to_sql(table_name, conn, if_exists="replace", index=False, method="multi")

            print(f"[CLOUD] Success: Table '{table_name}' updated in RDS")
        except Exception as e:
            raise RuntimeError(f"[ERROR] SQL Upload failed: {e}")

# ==========================================
# MAIN PIPELINE (note: dans un notebook, tu peux exécuter cellule par cellule)
# ==========================================

if __name__ == "__main__":
    cloud_manager = CloudManager(AWS_ACCESS_KEY, AWS_SECRET_KEY)

    df_weather.to_csv("weather_data.csv", index=False)
    df_hotels.to_csv("hotels_data.csv", index=False)

    cloud_manager.upload_to_s3("weather_data.csv", "weather/french_cities_weather.csv")
    cloud_manager.upload_to_s3("hotels_data.csv", "hotels/french_hotels.csv")

    cloud_manager.upload_to_sql(df_weather, "weather_cities")
    cloud_manager.upload_to_sql(df_hotels, "hotels_list")


### To go further if needed

In [ ]:
def pipeline_professionnel():
    """Version robuste pour la production"""
    import logging
    logger = logging.getLogger(__name__)
    
    # 1. Géocodage robuste
    df_geo = build_cities_geo_df()
    
    # 2. Météo avec parallélisation
    df_weather = build_weather_df(df_geo)
    
    # 3. Hôtels avec fallback
    df_hotels = build_hotels_df(df_weather)
    
    # 4. Sauvegarde sécurisée
    cloud_mgr = SecureCloudManager()
    cloud_mgr.upload_to_s3("weather_cities.csv", "raw/weather.csv")
    cloud_mgr.upload_to_sql(df_weather, "weather_forecasts")
    
    # 5. Visualisations
    plot_weather_animated_fixed(df_weather)
    plot_weather_filter(df_weather, metric="score")
    
    return df_geo, df_weather, df_hotels